# Reproduce the final submission

This notebook rebuilds `final_decision/results/submission_exp169_guarded.csv` from the weights stored in
this repository, without training anything. The pipeline:

1. **Mask2Former** (Swin-S backbone, exp139 epoch 40) proposes filament masks on each test photograph, seen
   eleven ways: as is, in the seven other flips/rotations, and shifted by 2 px three ways.
2. **Two YOLO models** (YOLOv8l from exp097, YOLO11m from exp002) give independent detections.
3. **exp169's vote** scores each Mask2Former mask by how consistently the views and the YOLO models
   agree, `score = s^0.3 x mean^0.7`, keeps scores of at least 0.68, and builds each mask from the
   pixel majority of its views.
4. **exp199's guard** puts back the large Mask2Former filaments (at least 3,000 px) that the vote
   removed, when nothing else covers them.

Each stage checks itself against a recorded number and stops on a mismatch. Set `DATA` to the competition
folder (the one that holds `train/` and `test/`). A GPU makes stages 2 and 3 take about 70 minutes instead of a
day or more on a CPU.

In [ ]:
import os, subprocess, sys
from pathlib import Path
here = Path.cwd().resolve()
# Inside the repository, or next to a clone of it (on Kaggle: git clone ... into /kaggle/working/Sol).
ROOT = next((p for p in (here, *here.parents, here / "Sol") if (p / "reproduce" / "run.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError(f"no clone of the repository in or next to {here}")
DATA = os.environ.get("SOL_DATA")   # the competition folder; when unset, run.py finds it under /kaggle/input
WORK = ROOT / "reproduce" / "work"
def stage(*names):
    subprocess.run([sys.executable, str(ROOT / "reproduce" / "run.py"), *(["--data", DATA] if DATA else []),
                    "--work", str(WORK), "--stages", *names], check=True)
print(ROOT, DATA, WORK)

## Environment

The versions the submission was made with (`requirements.txt` says where each comes from). torch and
torchvision come from the CUDA 12.8 index. Each stage below runs in its own process.

In [ ]:
# !pip uninstall -y -q torchaudio            # on Kaggle: built for another torch
# !pip install -q torch==2.8.0 torchvision==0.23.0 --index-url https://download.pytorch.org/whl/cu128
# !pip install -q -r {ROOT}/requirements.txt
import torch, transformers
print(torch.__version__, transformers.__version__, torch.cuda.device_count(), "GPU(s)")

## 1. Weights

GitHub stores files up to 100 MB, so checkpoints larger than 45 MiB are kept as numbered parts. This step
joins them and checks every part and every whole file against `weights/MANIFEST.json`.

In [ ]:
stage("weights")

## 2. Mask2Former candidates on the test photographs

Every candidate keeps its score (class probability times mean mask probability) and its mask, for the full
frame (floor 0.02) and the ten other views (floor 0.05), each mapped back to the original frame.

In [ ]:
stage("m2f")

## 3. YOLO pools

Both models run on the validation photographs first. Their pools must agree with the stored ones that exp169
was tuned on before the test pools are used.

In [ ]:
stage("yolo")

## 4. exp169's vote, then 5. the guard

The vote first replays its validation PQ (0.48894859995 on 106 photographs, 180 annotation records). It also
checks that the Mask2Former path rebuilds exp139's submitted file, then writes the test file. The guard
re-adds the 9 large filaments.

In [ ]:
stage("rule", "guard")

## 6. Check against the submitted file

In [ ]:
stage("check")
print((WORK / "final" / "check.json").read_text())